# LGT-Net demo setup

This single notebook supports both local Jupyter and Google Colaboratory. It reuses an existing local checkout when available; otherwise, it clones LGT-Net into `WORK_DIR/LGT-Net`. Google Drive dataset archives and published checkpoint archives are cached under `WORK_DIR/.downloads/`, extracted under the project directory, and executed through the locked uv environment.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import os
import shutil
import subprocess
import sys
import sysconfig
import zipfile

try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

WORK_DIR = Path.cwd().resolve()
DOWNLOAD_DIR = WORK_DIR / ".downloads"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
REPOSITORY_URL = "https://github.com/PINTO0309/LGT-Net.git"
RELEASE_BASE_URL = "https://github.com/PINTO0309/LGT-Net/releases/download/data"

def download_release_asset(asset_name):
    destination = DOWNLOAD_DIR / asset_name
    if destination.is_file():
        print(f"Using cached archive: {destination}")
        return destination

    partial = destination.with_suffix(destination.suffix + ".part")
    request = Request(
        f"{RELEASE_BASE_URL}/{asset_name}",
        headers={"User-Agent": "LGT-Net-demo"},
    )
    print(f"Downloading: {request.full_url}")
    with urlopen(request) as response, partial.open("wb") as output:
        shutil.copyfileobj(response, output)
    partial.replace(destination)
    return destination

def extract_archive(archive, destination):
    archive = Path(archive).resolve()
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zip_file:
        for member in zip_file.infolist():
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError:
                raise ValueError(f"Unsafe archive member: {member.filename}")
        zip_file.extractall(destination)
    print(f"Extracted {archive.name} to {destination}")

def extract_release_asset(asset_name, destination):
    extract_archive(download_release_asset(asset_name), destination)

if (WORK_DIR / "inference.py").is_file():
    PROJECT_DIR = WORK_DIR
elif (WORK_DIR / "LGT-Net" / "inference.py").is_file():
    PROJECT_DIR = WORK_DIR / "LGT-Net"
else:
    PROJECT_DIR = WORK_DIR / "LGT-Net"
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True,
    )

In [ ]:
os.chdir(PROJECT_DIR)
runtime_name = "Google Colab" if IN_COLAB else "local Jupyter"
print(f"Runtime: {runtime_name}")
print(f"Working directory: {PROJECT_DIR}")

## 1. Create the locked Python environment

LGT-Net is locked to CPython 3.12.12 and the dependency versions in `uv.lock`. The supported target is Linux x86_64 with glibc 2.31 or newer. CUDA 12.8 is supplied by the PyTorch wheels, so a system CUDA Toolkit is not required; GPU execution requires a compatible NVIDIA driver. On Colab, this cell installs a pinned uv bootstrap when uv is absent. For local Jupyter, install [uv](https://docs.astral.sh/uv/) first. The Notebook kernel itself does not need to use Python 3.12.12 because all project commands run in uv's managed environment.

In [ ]:
UV_BOOTSTRAP_VERSION = "0.9.26"
UV = shutil.which("uv")
if UV is None and IN_COLAB:
    print(f"Installing uv=={UV_BOOTSTRAP_VERSION} for the Colab runtime...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            f"uv=={UV_BOOTSTRAP_VERSION}",
        ],
        check=True,
    )
    UV = shutil.which("uv")
    if UV is None:
        uv_candidate = Path(sysconfig.get_path("scripts")) / "uv"
        if uv_candidate.is_file():
            UV = str(uv_candidate)

if UV is None:
    raise RuntimeError(
        "uv is required for local execution: "
        "https://docs.astral.sh/uv/getting-started/installation/"
    )

subprocess.run([UV, "sync", "--frozen"], cwd=PROJECT_DIR, check=True)
subprocess.run(
    [
        UV,
        "run",
        "python",
        "-c",
        (
            "import sys, numpy, torch, torchvision; "
            "print(f'Python {sys.version.split()[0]}'); "
            "print(f'torch {torch.__version__}, torchvision {torchvision.__version__}, '"
            "      f'numpy {numpy.__version__}, CUDA runtime {torch.version.cuda}, '"
            "      f'CUDA available {torch.cuda.is_available()}')"
        ),
    ],
    cwd=PROJECT_DIR,
    check=True,
)

def uv_run(*args):
    return subprocess.run([UV, "run", *args], cwd=PROJECT_DIR, check=True)

def download_google_drive_asset(file_id, asset_name):
    destination = DOWNLOAD_DIR / asset_name
    if destination.is_file():
        print(f"Using cached archive: {destination}")
        return destination

    partial = destination.with_suffix(destination.suffix + ".part")
    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"Downloading: {url}")
    uv_run("gdown", url, "--output", str(partial))
    partial.replace(destination)
    return destination

## 2. Prepare datasets

Run only the dataset cells needed for your task. Google Drive archives remain cached in `WORK_DIR/.downloads/`.

### MatterportLayout

Download the [MatterportLayout archive](https://drive.google.com/file/d/1rEWXy5zHVozHC0hKHsuHTxchrNHfYoJ4/view?usp=sharing) from Google Drive and extract it to `src/dataset/mp3d/`. This dataset is required by the MP3D and ablation-study evaluation and training examples.

In [ ]:
mp3d_archive = download_google_drive_asset(
    "1rEWXy5zHVozHC0hKHsuHTxchrNHfYoJ4",
    "mp3d.zip",
)
extract_archive(mp3d_archive, PROJECT_DIR / "src" / "dataset")

### PanoContext and Stanford 2D-3D

Download the [combined archive](https://drive.google.com/file/d/164DnSxz6ap8GcytRAPfJlIMvNPaikZEc/view?usp=sharing) from Google Drive and extract it to `src/dataset/pano_s2d3d/`. The `pano.yaml` and `s2d3d.yaml` configurations select their respective subsets from this directory.

In [ ]:
pano_s2d3d_archive = download_google_drive_asset(
    "164DnSxz6ap8GcytRAPfJlIMvNPaikZEc",
    "pano_s2d3d.zip",
)
extract_archive(pano_s2d3d_archive, PROJECT_DIR / "src" / "dataset")

### ZInD (optional)

The ZInD dataset is not included in the LGT-Net data release. Use the [official ZInD download tools](https://github.com/zillow/zind), then place the dataset at `src/dataset/zind/` before running ZInD evaluation or training.

## 3. Download pretrained checkpoints


Pretrained checkpoints are distributed from the [data release](https://github.com/PINTO0309/LGT-Net/releases/tag/data).

- [Benchmark checkpoints](https://github.com/PINTO0309/LGT-Net/releases/download/data/lgt-net-checkpoints-benchmarks.zip): MP3D, ZInD, PanoContext, and Stanford 2D-3D
- [Ablation Study checkpoint](https://github.com/PINTO0309/LGT-Net/releases/download/data/lgt-net-checkpoints-ablation-study.zip): Ours (full) on MatterportLayout

Both archives preserve the `checkpoints/SWG_Transformer_LGT_Net/` hierarchy and are extracted directly into `PROJECT_DIR`. The benchmark archive contains MP3D, ZInD, PanoContext, and Stanford 2D-3D checkpoints; the ablation archive is separate to keep each Release asset below GitHub's size limit.

In [ ]:
checkpoint_archives = [
    "lgt-net-checkpoints-benchmarks.zip",
    "lgt-net-checkpoints-ablation-study.zip",
]
for archive_name in checkpoint_archives:
    extract_release_asset(archive_name, PROJECT_DIR)

## 4. Run inference

This example uses the MP3D checkpoint and `src/demo/demo1.png`. It writes `demo1_pred.png`, `demo1_pred.json`, and the Manhattan-alignment `demo1_vp.txt` under `src/output/`. Inference automatically falls back to CPU when CUDA is unavailable.

In [ ]:
uv_run(
    "python",
    "inference.py",
    "--cfg",
    "src/config/mp3d.yaml",
    "--img_glob",
    "src/demo/demo1.png",
    "--output_dir",
    "src/output",
    "--post_processing",
    "manhattan",
)

from IPython.display import Image as NotebookImage, display

prediction_path = PROJECT_DIR / "src" / "output" / "demo1_pred.png"
display(NotebookImage(filename=str(prediction_path)))

## 5. Evaluate pretrained checkpoints

Each evaluation cell requires the corresponding extracted dataset and checkpoint. These commands use the metrics selected by the current README examples.

### MatterportLayout

The first cell evaluates the standard MP3D checkpoint. The second evaluates the full ablation-study configuration on the same dataset.

In [ ]:
uv_run("python", "main.py", "--cfg", "src/config/mp3d.yaml", "--mode", "test", "--need_rmse")

In [ ]:
uv_run("python", "main.py", "--cfg", "src/config/ablation_study/full.yaml", "--mode", "test", "--need_rmse")

### ZInD

This evaluation requires a separately downloaded ZInD dataset at `src/dataset/zind/`.

In [ ]:
uv_run("python", "main.py", "--cfg", "src/config/zind.yaml", "--mode", "test", "--need_rmse")

### PanoContext

In [ ]:
uv_run(
    "python", "main.py",
    "--cfg", "src/config/pano.yaml",
    "--mode", "test",
    "--need_cpe",
    "--post_processing", "manhattan",
    "--force_cube",
)

### Stanford 2D-3D

`src/config/s2d3d.yaml` currently selects `cuda:2`. Change `TRAIN.DEVICE` in a copied configuration if that device index is unavailable on your system.

In [ ]:
uv_run(
    "python", "main.py",
    "--cfg", "src/config/s2d3d.yaml",
    "--mode", "test",
    "--need_cpe",
    "--post_processing", "manhattan",
    "--force_cube",
)

## 6. Train

The example below trains with `src/config/mp3d.yaml`. That configuration has `TRAIN.SCRATCH: false`, so an existing MP3D checkpoint is used as initialization. Copy and adjust the YAML, checkpoint tag, device, batch size, and scratch/resume settings before starting a new experiment. Training is configured for CUDA and is a long-running operation.

In [ ]:
uv_run("python", "main.py", "--cfg", "src/config/mp3d.yaml", "--mode", "train")